In [11]:
#from skimage import feature
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
from scipy import signal
import os
from pathlib import Path
import sys
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, average_precision_score
from sklearn.metrics import precision_recall_curve

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier as DTC
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.neighbors import KNeighborsClassifier as KNN
import xgboost as XGB


In [12]:
parent_folder = Path().resolve().parent
src_path = parent_folder / 'src'
sys.path.append(str(src_path))

from tools import get_embedding_birdnet

#env to use: clef

In [21]:
root_folder='../data/Holderried/selections_wavs/'
df_pos1 = pd.read_csv(root_folder + 'birdnet_probs/1/predictions.csv')

df_pos1["selec"] = df_pos1["filename"].str.split(".").str[0]
df_pos1["selec"] = df_pos1["selec"].astype(int)

In [22]:
df_pos1.selec[0]

10

In [23]:
df_train = pd.read_csv("Holderried_pos_train_1956.csv") # Holder selections for training

In [24]:
df_pos1 = df_pos1.merge(
    df_train[["selec", "sound.files", "recording.id"]],
    on="selec",
    how="left"
)

df_pos1 = df_pos1.dropna() # this removes extra "selec" columns

In [25]:
len(df_pos1)

1935

In [26]:
df_pos1.head(1)

,filename,confidence,selec,sound.files,recording.id
1,1000.wav,0.992285,1000,20200625_194506.WAV,15562.0


In [27]:
root_folder='../data/XC_long/' # XC Long positive records
df_pos2 = pd.read_csv(root_folder + 'birdnet_probs/1/predictions.csv')
df_pos2["wavname"] = df_pos2["filename"].str.split("-").str[0].str[:-1]


In [118]:
root_folder='../data/train_data/xc/' # XC Inintial Submission positive records
df_pos3 = pd.read_csv(root_folder + 'birdnet_probs/1/predictions.csv')
df_pos3["filename"] = df_pos3["filename"].str.split("-").str[0].str[:-1]

In [121]:
df_pos = pd.concat([df_pos1, df_pos2, df_pos3], ignore_index=True)
#df_pos = pd.concat([df_pos1, df_pos2], ignore_index=True)

In [122]:
len(df_pos)

3054

In [34]:
root_folder='../data/FVA_non_Eurasian_segs/seed2/'
df_neg1 = pd.read_csv(root_folder + 'birdnet_probs/0/predictions.csv')
df_neg1["filename"] = df_neg1["filename"].str.split("_").str[0]

In [36]:
root_folder='../data/XC_long/' # XC Long positive records
df_neg2 = pd.read_csv(root_folder + 'birdnet_probs/0/predictions.csv')
df_neg2["filename"] = df_neg2["filename"].str.split("-").str[0].str[:-1]

In [84]:
root_folder='../data/train_data/embedding/northwest_DE/' # North DE records
df_neg3 = pd.read_csv(root_folder + 'birdnet_probs/0/predictions.csv')
df_neg3["filename"] = df_neg3["filename"].str.split("_").str[0]

In [85]:
df_neg3.filename[0]

'3dBhE6'

In [100]:
root_folder='../data/audiomoth_DE/' # Audiomoth DE negative records
df_neg4 = pd.read_csv(root_folder + 'birdnet_probs/0/predictions.csv')
df_neg4["filename"] = df_neg4["filename"].str.split("_").str[0]

In [112]:
root_folder='../data/audiomoth_Eurasian/' # Audimoth negative on Eurasian genus (except woodcock) 
df_neg5 = pd.read_csv(root_folder + 'birdnet_probs/0/predictions.csv')
df_neg5["filename"] = df_neg5["filename"].str.split("_").str[0]

In [39]:
root_folder='../data/train_data_5/seed1/' # Baden-Wurttemberg negative records
df_neg6 = pd.read_csv(root_folder + 'birdnet_probs/0/predictions.csv')
df_neg6["filename"] = df_neg6["filename"].str.split("_").str[0]

In [113]:
df_neg = pd.concat([df_neg1, df_neg2, df_neg3, df_neg4, df_neg5, df_neg6], ignore_index=True)
#df_neg = pd.concat([df_neg1, df_neg2, df_neg3, df_neg4, df_neg6], ignore_index=True)

In [98]:
#df_neg = df_neg[df_neg.confidence > 0.0]

In [114]:
len(df_neg)

16122

In [123]:
df_pos['target'] = 1
df_neg['target'] = 0

In [130]:
from sklearn.model_selection import GroupKFold, cross_val_score

# Combine and shuffle
df_combined = pd.concat([df_pos, df_neg], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=22).reset_index(drop=True)

# Features, labels, groups
X = df_combined["confidence"].values
y = df_combined["target"].values
groups = df_combined["filename"].values

# 5-fold grouped CV
cv = GroupKFold(n_splits=5)

In [128]:
from sklearn.metrics import accuracy_score, average_precision_score

y_true = df_combined["target"]
y_score = df_combined["confidence"]

# Accuracy (using 0.5 as classification threshold)
y_pred = (y_score >= 0.5).astype(int)
accuracy = accuracy_score(y_true, y_pred)

# Average Precision (uses continuous confidence scores)
ap = average_precision_score(y_true, y_score)

print(f"Accuracy: {accuracy:.4f}")
print(f"Average Precision: {ap:.4f}")

Accuracy: 0.9344
Average Precision: 0.8349


### Cross-Validation

In [131]:
# Assign fold to every sample
df_combined["fold"] = -1

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y, groups)):
    df_combined.loc[val_idx, "fold"] = fold

BirdNET

In [132]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, average_precision_score
import numpy as np
import pandas as pd

# 5-fold grouped CV
cv = GroupKFold(n_splits=5)

accuracies = []
average_precisions = []

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):

    y_true = y[test_idx]
    y_score = X[test_idx]

    # Accuracy using threshold 0.5
    y_pred = (y_score >= 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred)

    # Average Precision
    ap = average_precision_score(y_true, y_score)

    accuracies.append(acc)
    average_precisions.append(ap)

    print(f"Fold {fold}: Accuracy = {acc:.4f}, AP = {ap:.4f}")

print("\n===== 5-Fold Results =====")
print(f"Accuracy: {np.mean(accuracies):.4f} ± {np.std(accuracies, ddof=1):.4f}")
print(f"Average Precision: {np.mean(average_precisions):.4f} ± {np.std(average_precisions, ddof=1):.4f}")

Fold 1: Accuracy = 0.9359, AP = 0.8326
Fold 2: Accuracy = 0.9372, AP = 0.8365
Fold 3: Accuracy = 0.9312, AP = 0.8442
Fold 4: Accuracy = 0.9327, AP = 0.8220
Fold 5: Accuracy = 0.9351, AP = 0.8407

===== 5-Fold Results =====
Accuracy: 0.9344 ± 0.0024
Average Precision: 0.8352 ± 0.0086
